# 03 · export v3 — Z: files → per-split canonical parquet

**Kernel: `fttl-v3` (env-v3, Python 3.11, xgboost 3.2.0).** v3's raw extract and transformed
data exist on `Z:` (confirmed 2026-08-10 — the training script persists nothing, but the data
is there anyway), already split **train / test / oot** (confirmed 2026-08-12). The transformed
files carry the train-time predictions as columns (`fttl_predicted_prob` /
`fttl_predicted_label`), so scores are **taken, not recomputed** — a sample re-score only
verifies that the Z: files and the shipped pickle come from the same run.

**Fill in on the company laptop: only the `Z:` paths.** Column names come from `config`
(v3 values filled on the company machine).

| writes (× train/test/oot) | canonical columns |
|---|---|
| `inputs/raw_v3.parquet` *(single file)* | raw extract as-is, id renamed (`ID_CLAIM` stays beside it) |
| `inputs/features_v3_{split}.parquet` | `claim_id` + transformed matrix (target, saved scores kept) |
| `inputs/targets_v3_{split}.parquet` | `claim_id, date, observed` |
| `detection/v3_scores_{split}.parquet` | `claim_id, model_v3_score` *(saved train-time; in-sample for train)* |

**Key = `Claimnumber_CLAIM`, the business claim number** (`config.column("v3", "claim_id")`), the
field v2's data and the v2 serving log join on. It is *not* the repo's `project_params.KEY`
(`ID_CLAIM`, a database row id): the first export used that and every cross-version join
silently mismatched (2026-09-03; the files then on disk were re-keyed in place by
`src/data/rekey_v3_claim_id.py`). If a transformed file carries only `ID_CLAIM`, the claim
number is bridged in from the raw extract, which carries both.

Resolve these downstream via `config.split_path(kind, "v3", split)`; splits are
`config.SPLITS["v3"]`. Because these files are what training **actually saw**, the export
carries **no** as-of-now enrichment caveat — that applies only to the DB-regeneration
fallback route (README § "Training Flow").

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import config                      # noqa: E402

import xgboost                     # noqa: E402
assert xgboost.__version__ == config.xgboost_pin("v3"), (
    xgboost.__version__, "expected", config.xgboost_pin("v3"))

# ---- SOURCES: fill the real Z: paths. They live HERE, not in config — a declared config
# ---- path would point the analysis .venv at files it may not be able to open. ----------
RAW = r"Z:\...\<v3 raw extract>"
TRANSF = {
    "train": r"Z:\...\<train transformed>",
    "test":  r"Z:\...\<test transformed>",
    "oot":   r"Z:\...\<oot transformed>",
}
assert tuple(TRANSF) == config.SPLITS["v3"]

ID = config.column("v3", "claim_id")     # ID_CLAIM
DATE, OBSERVED = config.column("v3", "date"), config.column("v3", "observed")
SCORE = config.column("v3", "score")     # fttl_predicted_prob — saved train-time score

def read_table(path):
    "Suffix dispatch — v3 is a Polars repo, so Z: files may be parquet rather than pickle."
    path = str(path)
    if path.endswith(".pkl"):
        return joblib.load(path)     # Z: .pkl files are joblib dumps (every version so far)
    if path.endswith(".csv"):
        return pd.read_csv(path)
    return pd.read_parquet(path)


In [ ]:
raw = read_table(RAW)
print("raw", raw.shape)
assert ID in raw.columns, list(raw.columns)
assert raw[ID].is_unique, "duplicate claim numbers in the raw extract"

# ID is the BUSINESS claim number (Claimnumber_CLAIM) — see the header. The transformed files may
# carry only the repo's row key (ID_CLAIM); then the claim number is bridged in from raw, which
# carries both. Nothing is guessed: a split with neither column stops here.
ROW_KEY = "ID_CLAIM"
bridge = raw[[ROW_KEY, ID]] if ROW_KEY in raw.columns else None

D = {}
for s in config.SPLITS["v3"]:
    d = read_table(TRANSF[s])
    if ID not in d.columns:
        assert bridge is not None and ROW_KEY in d.columns, (
            s + ": has neither '" + ID + "' nor '" + ROW_KEY + "' (to bridge from raw)")
        n0 = len(d)
        d = d.merge(bridge, on=ROW_KEY, how="left", validate="one_to_one")
        assert len(d) == n0 and d[ID].notna().all(), (
            s + ": " + str(int(d[ID].isna().sum())) + " rows have no claim number in raw")
        print(s + ": " + ID + " bridged from raw via " + ROW_KEY)
    for col in (ID, DATE, OBSERVED, SCORE):
        assert col in d.columns, s + ": '" + col + "' not in the transformed file"
    assert d[ID].is_unique, s + ": duplicate claim ids"
    D[s] = d
    print(s, d.shape)
assert pd.concat([D[s][ID] for s in D]).is_unique, "splits overlap on claim id"

In [ ]:
# Integrity check: re-score a sample through p146_model.pkl and compare with the saved
# column. A match proves model pkl + matrix + env line up; a mismatch means the Z: files
# and the pickle are from DIFFERENT runs — stop and investigate before exporting.
est = joblib.load(config.path("model", "v3", "real"))     # needs repo_dir declared in config
booster_cols = list(est.get_booster().feature_names)
proc = pd.concat([D[s] for s in config.SPLITS["v3"]], ignore_index=True)
missing = [c for c in booster_cols if c not in proc.columns]
if missing:
    print("cannot verify — model features absent from the matrix:", missing[:8])
else:
    idx = np.random.RandomState(0).choice(len(proc), size=min(2000, len(proc)), replace=False)
    recomputed = est.predict_proba(proc.iloc[idx][booster_cols])[:, 1]
    gap = float(np.max(np.abs(recomputed - proc.iloc[idx][SCORE].values)))
    print("max |recomputed - saved| =", gap, " (OK)" if gap < 1e-5 else " *** MISMATCH ***")


In [ ]:
# ---- per-split export: features / targets / scores, one parquet each -------------------
# No format conversion here: v3's Z: raw + transformed files are ALREADY parquet, so
# read_table hands back Arrow-typed frames and arrow_safe below is a no-op. It stays only
# because read_table also accepts .csv/.pkl — those routes can produce object columns that
# mix python types (v2's AirbagsDeployed: True/true/Yes/Y/False/false/No/N) which Arrow
# refuses to write. Word forms only, so a "1"/"0" string column is never silently retyped.
BOOLWORDS = {"true": True, "false": False, "t": True, "f": False,
             "yes": True, "no": False, "y": True, "n": False}

def _as_bool(v):
    # pd.api.types.is_bool, NOT isinstance(v, bool): numpy bool_ is not a bool subclass and
    # would otherwise leak the whole column into the string branch.
    if pd.api.types.is_bool(v):
        return bool(v)
    if isinstance(v, str) and v.strip().lower() in BOOLWORDS:
        return BOOLWORDS[v.strip().lower()]
    return None

def arrow_safe(df):
    """Object columns Arrow can type: boolean-ish -> nullable boolean, anything else still
    mixing python types -> nullable string. Values are preserved. Copies only if something
    actually needs fixing — v3's matrices are large and the parquet path fixes nothing."""
    fixes = {}
    for c in df.columns[df.dtypes == "object"]:
        s = df[c]
        vals = s.dropna().unique()
        if len(vals) and all(_as_bool(v) is not None for v in vals):
            fixes[c] = s.map(lambda v: _as_bool(v) if pd.notna(v) else pd.NA).astype("boolean")
            # print the SPELLING counts, not just the set: which encodings dominate is a
            # data-provenance signal (source system / era), worth seeing before it is erased.
            print(f"    {c}: object -> boolean   {s.value_counts(dropna=False).to_dict()}")
        elif s.map(type).nunique(dropna=True) > 1:
            fixes[c] = s.astype("string")
            print(f"    {c}: mixed object -> string   "
                  f"types {sorted(t.__name__ for t in s.map(type).unique())}")
    return df.assign(**fixes) if fixes else df

WRITTEN = []
def write(df, p):
    p.parent.mkdir(parents=True, exist_ok=True)
    df = arrow_safe(df)
    df.to_parquet(p, index=False)
    WRITTEN.append((p, len(df), list(df.columns)))
    print(f"wrote {p.relative_to(config.ROOT)}  rows {len(df):>9}")

write(raw.rename(columns={ID: "claim_id"}), config.path("raw_dataset", "v3", "real"))

for s in config.SPLITS["v3"]:
    d = D[s]

    # features keep ALL columns (incl. fttl_predicted_prob/label, target, and ID_CLAIM when the
    # transformed file carries it) — consumers set aside non-model columns themselves against
    # the booster's own feature list.
    write(d.rename(columns={ID: "claim_id"}),
          config.split_path("processed_inputs", "v3", s))

    write(d[[ID, DATE, OBSERVED]].rename(
              columns={ID: "claim_id", DATE: "date", OBSERVED: "observed"}),
          config.split_path("targets", "v3", s))

    # the SAVED train-time scores — the actual training run's own numbers, not recomputed
    write(d[[ID, SCORE]].rename(columns={ID: "claim_id", SCORE: "model_v3_score"}),
          config.split_path("scores", "v3", s))

In [ ]:
# ---- confirm export: re-read every parquet's metadata, rows + columns vs in-memory ------
import pyarrow.parquet as pq

for p, n_exp, cols_exp in WRITTEN:
    name = str(p.relative_to(config.ROOT))
    n = pq.read_metadata(p).num_rows
    cols = list(pq.read_schema(p).names)
    print(f"{name:<52} {p.stat().st_size / 1024**2:>9.1f} MB  rows {n:>9}")
    assert n == n_exp, f"{name}: rows {n} != in-memory {n_exp}"
    assert cols == cols_exp, f"{name}: column mismatch"
print(f"\nall {len(WRITTEN)} parquet files OK")


Notes:
- **No log cell on purpose**: v3 was never deployed, so no production log exists — reconfirmed
  2026-08-10. Anything needing real decisions must come from v2's log.
- `fttl_predicted_label` (score > tuned threshold) is deliberately NOT exported as a canonical
  `decision`: canonical decision means the treatment actually applied to a car, and v3 never
  actioned anything. It stays available inside the features parquet.
- The saved scores are in-sample for the train split.
- If the verify cell reports a mismatch, the Z: files predate (or postdate) the shipped pickle —
  fall back to the DB regeneration chain and carry its enrichment caveat.
- **Leave `paths.processed_inputs` / `paths.raw_dataset` undeclared in config** — analysis reads
  the parquet this notebook wrote at the fallback paths; that is the design.
